# Full Breakout DQN training — arXiv 2013 (Google Colab)

[Open in Google Colab](https://colab.research.google.com/github/benphuang/atari/blob/main/atari_arxiv2013.ipynb).

Select a GPU and high-RAM runtime, then run the cells in order. This notebook trains the `arxiv2013` preset for its full **10,000,000 agent steps**, using the preset's algorithm and evaluation settings. It stores TensorBoard logs and the final model in Google Drive. One-frame replay uses about **6.6 GiB of host RAM** at the preset's one-million-entry capacity; it is not stored in VRAM or on Drive.

**Runtime warning:** Colab resource limits and session duration are not guaranteed ([Colab FAQ](https://research.google.com/colaboratory/faq.html#resource-limits)). Checkpointing is available in the training CLI, but this Colab notebook does not enable it. For resumable training on the Rutgers iLab machine, use the commands in [CHECKPOINTING.md](https://github.com/benphuang/atari/blob/main/CHECKPOINTING.md).

**ROM terms:** ROM installation is a separate, interactive step. Review and accept the AutoROM license before entering `ACCEPT`. Colab's `/content` storage is temporary.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/benphuang/atari.git"
PROJECT_DIR = Path("/content/atari")

if (PROJECT_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
elif PROJECT_DIR.exists():
    raise FileExistsError(f"{PROJECT_DIR} exists but is not a git checkout; remove or rename it and retry.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)

# Keep Colab's preinstalled NumPy and PyTorch builds.
packages = [
    "ale-py>=0.11",
    "AutoROM>=0.6",
    "gymnasium>=1.0,<2",
    "opencv-python-headless>=4.8",
    "tensorboard>=2.14",
    "tyro>=0.8",
]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)

import torch
print(f"Python: {sys.version.split()[0]} | PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU unavailable; the full training cell requires a GPU runtime.")

## Install Atari ROMs

Review the AutoROM license before running this cell. The installation did not auto-accept the license.

In [ ]:
import shutil

if input("After reviewing the AutoROM license, type ACCEPT to download ROMs: ").strip() != "ACCEPT":
    raise RuntimeError("ROM installation skipped; accept the license before rerunning this cell.")
autorom = shutil.which("AutoROM")
if autorom is None:
    raise RuntimeError("AutoROM executable was not installed in this runtime.")
subprocess.run([autorom, "--accept-license"], check=True)

In [ ]:
import os
import sys

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import ale_py
import gymnasium as gym

gym.register_envs(ale_py)
env = gym.make("BreakoutNoFrameskip-v4")
observation, _ = env.reset(seed=1)
assert observation.ndim == 3 and observation.shape[-1] == 3, f"Unexpected observation shape: {observation.shape}"
print(f"Breakout preflight passed: {observation.shape}, {env.action_space.n} actions")
env.close()

## Run full arXiv 2013 training

The preset's full 10-million-step budget, network, loss, optimizer, preprocessing, frame-skip, exploration, learning-start, and evaluation settings are preserved. The only operational override is frame-based replay storage, which uses less host RAM than the preset's stacked replay layout.

In [ ]:
from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")
drive.mount(str(DRIVE_MOUNT))
MY_DRIVE = DRIVE_MOUNT / "MyDrive"
if not MY_DRIVE.is_dir():
    raise FileNotFoundError(f"Google Drive is not mounted at {MY_DRIVE}.")

PRESET_NAME = "arxiv2013"
EXP_NAME = "colab_arxiv2013_breakout_full"
OUTPUT_DIR = MY_DRIVE / "atari_dqn_runs" / PRESET_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Training logs and final model will be written under: {OUTPUT_DIR}")

In [ ]:
import os
import runpy
import sys
from pathlib import Path

import torch

PROJECT_DIR = Path("/content/atari").resolve()
TRAINING_SCRIPT = PROJECT_DIR / "atari_mnih.py"
if not TRAINING_SCRIPT.is_file():
    raise FileNotFoundError(f"{TRAINING_SCRIPT} is missing. Run the setup cell first.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
if not torch.cuda.is_available():
    raise RuntimeError("A GPU is required for this full run. Change the Colab runtime type to a GPU and rerun.")
os.chdir(OUTPUT_DIR)

TOTAL_TIMESTEPS = 10_000_000
training_args = [
    "--preset", "arxiv2013",
    "--exp-name", EXP_NAME,
    "--save-model",
    "--replay-storage", "frames",
]

print(f"Starting full arxiv2013 run: {TOTAL_TIMESTEPS:,} steps on {torch.cuda.get_device_name(0)}")
previous_argv = sys.argv
try:
    sys.argv = [str(TRAINING_SCRIPT), *training_args]
    runpy.run_path(str(TRAINING_SCRIPT), run_name="__main__")
finally:
    sys.argv = previous_argv

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

run_dirs = sorted((OUTPUT_DIR / "runs").glob(f"BreakoutNoFrameskip-v4__{EXP_NAME}__1__*"), key=lambda path: path.stat().st_mtime)
assert run_dirs, f"No training run found under {OUTPUT_DIR / 'runs'}"
run_dir = run_dirs[-1]
model_path = run_dir / f"{EXP_NAME}.cleanrl_model"
assert model_path.is_file(), f"Final model was not saved: {model_path}"
event_files = sorted(run_dir.glob("events.out.tfevents.*"))
assert event_files, "No TensorBoard event file was written."
events = EventAccumulator(str(event_files[-1])).Reload()
assert "eval/mean_return" in events.Tags()["scalars"], f"Missing final evaluation metric: {events.Tags()}"
evaluation = events.Scalars("eval/mean_return")[-1]
assert evaluation.step == TOTAL_TIMESTEPS, f"Expected final evaluation at {TOTAL_TIMESTEPS}; got {evaluation.step}"
episodes = events.Scalars("eval/num_episodes")[-1].value
partial = bool(events.Scalars("eval/partial")[-1].value)
print(f"Full arXiv 2013 run finished. Final eval mean={evaluation.value:.2f}, episodes={episodes:.0f}, partial={partial}.")
print(f"Model: {model_path} ({model_path.stat().st_size / 2**20:.1f} MiB)")